# AERA Recovery (Android 16) - Google Colab 极速编译方案

**优势**：Google Colab 免费提供 **107GB 纯净 SSD** 硬盘空间与 **12GB+ RAM**，网络直连 Google/GitHub（下载速度 100MB/s+），彻底消除 GitHub Actions 的磁盘爆满与内存溢出瓶颈。

### 执行步骤：
1. 点击顶部菜单：**代码执行程序 -> 更改运行时类型 -> 选择 CPU（或者标准运行时）**
2. 依次按 `Shift + Enter` 运行下方各个代码块。

In [ ]:
# 1. 检查可用磁盘与系统资源
!df -h / 
!free -h
!lscpu | grep 'Model name\|CPU(s):'

In [ ]:
# 2. 安装 Android 16 编译依赖与 repo 工具
!sudo apt-get update -qq
!sudo apt-get install -y -qq git-core gnupg flex bison build-essential zip curl zlib1g-dev \
  libc6-dev-i386 libncurses5 lib32ncurses5-dev x11proto-core-dev libx11-dev \
  lib32z1-dev libgl1-mesa-dev libxml2-utils xsltproc unzip fontconfig \
  python3 python-is-python3 rsync lz4 bc ccache

!mkdir -p /root/bin
!curl -s https://storage.googleapis.com/git-repo-downloads/repo > /root/bin/repo
!chmod a+x /root/bin/repo
!sudo ln -sf /root/bin/repo /usr/bin/repo
!git config --global user.name "AERA-Colab-Builder"
!git config --global user.email "builder@colab.local"
!git config --global url."https://github.com/".insteadOf "ssh://git@github.com/"
!git config --global url."https://github.com/".insteadOf "git@github.com:"

In [ ]:
# 3. 初始化并同步 AERA 16.0 源码 (仅保留 depth=1)
%cd /content
!mkdir -p aera_workspace
%cd /content/aera_workspace

!repo init --depth=1 -u https://github.com/AERA-Recovery/android_manifest.git -b aera-16.0
!find .repo/manifests -name "*.xml" -exec sed -i 's|ssh://git@github.com/|https://github.com/|g' {} +
!repo sync -c --no-clone-bundle --no-tags --optimized-fetch --prune -j$(nproc)

# 丢卒保车：删除 .repo 历史对象，腾出 20GB+ 空间
!rm -rf .repo
!df -h /content

In [ ]:
# 4. 克隆 OnePlus Ace 5 / Giulia Device Tree
%cd /content/aera_workspace
!git clone --depth=1 https://github.com/cheng343/android_device_oneplus_giulia-AERA -b aera-16.0 device/oneplus/giulia

In [ ]:
# 5. 执行 Android 16 编译
%cd /content/aera_workspace
import os
os.environ['GOGC'] = '50'
os.environ['GOMEMLIMIT'] = '10GiB'
os.environ['ALLOW_MISSING_DEPENDENCIES'] = 'true'

# 加载环境并编译 recoveryimage
!bash -c 'source build/envsetup.sh && lunch twrp_giulia-bp2a-eng && m recoveryimage'

In [ ]:
# 6. 归集编译出的 recovery.img 并直接下载到本地
from google.colab import files
import glob

imgs = glob.glob('/content/aera_workspace/out/target/product/giulia/*.img')
print('Found recovery images:', imgs)
for f in imgs:
    print(f'Downloading {f}...')
    files.download(f)